# Assignment 1: AI-Powered Intrusion Detection Solution

**Course:** Information Security\n**Aligned CLO:** 4 - Create solutions to real life scenarios using different security related tools.\n
## 1. Objective & Real-Life Scenario
As a security analyst at SecureNet Corp, the objective is to develop a Machine Learning model to augment the existing Network Intrusion Detection System (NIDS). This model will classify network traffic as `BENIGN` or malicious (e.g., `DDoS`, `Brute_Force`, `Web_Attack`, `Infiltration`).

## Step 1: Problem Definition & Tool Setup
We are using a synthetic dataset generator modeled after the **CIC-IDS2017** dataset. This dataset was chosen as it contains realistic DDoS, Brute-Force, and Web Attack traffic, which are common threats to organizational web servers.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Import our custom data generator
import sys
sys.path.append('.')
from utils.data_generator import generate_ids_dataset

# Generate realistic network traffic data
df = generate_ids_dataset(n_samples=10000, random_state=42)
print(f"Dataset Shape: {df.shape}")
df.head()

## Step 2: Data Preprocessing & EDA
Security analysts must clean and prepare raw logs before analysis. We will handle missing values, encode categorical variables, and perform Exploratory Data Analysis.

In [ ]:
# EDA: Class Distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Label')
plt.title('Distribution of Network Traffic Types')
plt.show()

print(df['Label'].value_counts(normalize=True) * 100)

In [ ]:
# Preprocessing: Encoding and Scaling
X = df.drop('Label', axis=1)
y_raw = df['Label']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing complete.")

## Step 3: Creating the ML Solution
We will design a **Random Forest Classifier**. 
*Justification*: Random Forest was selected for its high accuracy, robustness against overfitting, and ability to handle the complex, non-linear feature relationships commonly found in high-dimensional network traffic.

In [ ]:
# Initialize and Train Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
print("Model training complete.")

## Step 4: Evaluating the Solution
We evaluate the model using accuracy, precision, recall, and a confusion matrix to understand its security effectiveness.

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Metrics
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

## Critical Security Analysis
The Random Forest model demonstrates excellent overall accuracy. Most importantly, we must look at the **Recall** for attack classes (like Infiltration or Web_Attack). 

If recall for an attack is low, it means our solution is allowing unauthorized access attempts to go undetected (False Negatives), which is a critical flaw. High recall ensures that real threats are flagged for security analysts to review, even if it introduces some False Positives (which impact precision).